In [ ]:

%pip install pandas numpy matplotlib seaborn ipykernel


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")


In [ ]:

from pathlib import Path

# Find the cleaned dataset
filename = "ecommerce_logistics_cleaned.csv"
base = Path.cwd()

possible_paths = [
    base / "Dataset" / "Cleaned" / filename,
    base.parent / "Dataset" / "Cleaned" / filename,
    base / "E-Commerce and Logistics delivery Analysis" / "Dataset" / "Cleaned" / filename,
]

file_path = next((p for p in possible_paths if p.is_file()), None)

if file_path is None:
    raise FileNotFoundError("Cleaned dataset not found. Check its folder location.")

# Load the cleaned dataset
df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully!")
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

display(df.head())


In [ ]:

# STEP 9: Data Overview and Quality Check

print("DATASET INFORMATION")
print("-" * 40)

print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nDATA TYPES:")
print(df.dtypes.to_string())

print("\nMISSING VALUES (TOP 15):")
missing = df.isnull().sum().sort_values(
    ascending=False
)
print(missing.head(15))

print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())

print("\nDELIVERY STATUS:")
print(df["is_late"].value_counts(dropna=False))

print("\nCANCELLED STATUS:")
print(df["is_cancelled"].value_counts(dropna=False))

print("\nKPI ELIGIBILITY:")
print(df["kpi_eligible"].value_counts(dropna=False))


In [ ]:

# STEP 10: Delivery Performance Analysis

# Select records eligible for KPI analysis
delivery_df = df[df["kpi_eligible"] == 1].copy()

# Check delivery status consistency within each order
status_check = delivery_df.groupby("Order Id")["is_late"].nunique()
print("Orders with mixed delivery statuses:",
      (status_check > 1).sum())

# One record per order to avoid counting order items multiple times
orders_df = delivery_df.drop_duplicates(subset="Order Id")

# Calculate delivery counts
delivery_counts = (
    orders_df["is_late"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

on_time = delivery_counts[0]
late = delivery_counts[1]
total = delivery_counts.sum()

late_percentage = (late / total) * 100 if total else 0

print("Total Eligible Orders:", total)
print("Not Late Orders:", on_time)
print("Late Orders:", late)
print(f"Late Delivery Percentage: {late_percentage:.2f}%")

# Create bar chart
plt.figure(figsize=(8, 5))

bars = plt.bar(
    ["Not Late", "Late"],
    [on_time, late],
    color=["seagreen", "tomato"]
)

plt.title("Delivery Performance Analysis")
plt.xlabel("Delivery Status")
plt.ylabel("Number of Orders")

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom"
    )


plt.tight_layout()

plt.savefig(
    visualizations_dir / "01_delivery_performance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



In [ ]:

# STEP 11: Shipping Mode vs Late Delivery Analysis

# Group unique eligible orders by shipping mode
shipping_analysis = orders_df.groupby(
    "Shipping Mode"
).agg(
    total_orders=("Order Id", "nunique"),
    late_orders=("is_late", "sum")
).reset_index()

# Calculate late delivery percentage
shipping_analysis["late_percentage"] = (
    shipping_analysis["late_orders"] /
    shipping_analysis["total_orders"] * 100
)

# Sort by highest late delivery rate
shipping_analysis = shipping_analysis.sort_values(
    by="late_percentage",
    ascending=False
)

print("SHIPPING MODE ANALYSIS")
display(shipping_analysis.round(2))

# Create bar chart
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=shipping_analysis,
    x="Shipping Mode",
    y="late_percentage"
)

plt.title("Late Delivery Rate by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Late Delivery Percentage (%)")
plt.ylim(0, 100)

# Display percentages on bars
for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(
        f"{height:.2f}%",
        (
            bar.get_x() + bar.get_width() / 2,
            height
        ),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()


In [ ]:

# STEP 12: Delivery Delay Days Analysis

# Use unique KPI-eligible orders
delay_df = orders_df.copy()

# Convert delay days into numeric values
delay_df["delay_days"] = pd.to_numeric(
    delay_df["delay_days"],
    errors="coerce"
)

# Verify delay flag consistency
mismatch = (
    (delay_df["delay_days"] > 0).astype(int)
    != delay_df["is_late"]
).sum()

print("Delay Flag Mismatches:", mismatch)

# Select late deliveries
late_df = delay_df[
    (delay_df["is_late"] == 1) &
    (delay_df["delay_days"].notna())
].copy()

# Calculate delay statistics
print("\nDELIVERY DELAY STATISTICS")

print("Total Late Orders:", len(late_df))

print(
    "Average Delay:",
    round(late_df["delay_days"].mean(), 2),
    "days"
)

print(
    "Median Delay:",
    round(late_df["delay_days"].median(), 2),
    "days"
)

print(
    "Maximum Delay:",
    late_df["delay_days"].max(),
    "days"
)

# Delay distribution
plt.figure(figsize=(10, 6))

sns.histplot(
    data=late_df,
    x="delay_days",
    discrete=True,
    color="steelblue"
)

plt.title("Distribution of Delivery Delays")
plt.xlabel("Delivery Delay (Days)")
plt.ylabel("Number of Late Orders")

plt.tight_layout()
plt.show()


In [ ]:

# STEP 13: Actual vs Scheduled Shipping Days

# Analyze shipping performance by mode
shipping_days = delay_df.groupby("Shipping Mode").agg(
    total_orders=("Order Id", "nunique"),
    avg_actual_days=("Days for shipping (real)", "mean"),
    avg_scheduled_days=("Days for shipment (scheduled)", "mean"),
    avg_delay_days=("delay_days", "mean"),
    late_rate=("is_late", "mean")
)

shipping_days["late_rate"] *= 100

print("SHIPPING PERFORMANCE COMPARISON")
display(shipping_days.round(2))

# Actual vs Scheduled comparison chart
shipping_days[
    ["avg_actual_days", "avg_scheduled_days"]
].plot(
    kind="bar",
    figsize=(10, 6),
    color=["steelblue", "orange"]
)

plt.title("Actual vs Scheduled Shipping Days")
plt.xlabel("Shipping Mode")
plt.ylabel("Average Shipping Days")
plt.xticks(rotation=0)
plt.legend(["Actual Days", "Scheduled Days"])
plt.tight_layout()
plt.show()


In [ ]:

# STEP 14: Monthly Order Trends Analysis

# Select one record per order
order_trends = df.drop_duplicates(
    subset="Order Id"
).copy()

# Convert order date into datetime
order_trends["order_date"] = pd.to_datetime(
    order_trends["order date (DateOrders)"],
    format="mixed",
    errors="coerce"
)

# Check date conversion
print("Invalid Dates:",
      order_trends["order_date"].isna().sum())

# Remove records with invalid dates
order_trends = order_trends.dropna(
    subset=["order_date"]
)

# Extract year and month
order_trends["order_month"] = (
    order_trends["order_date"]
    .dt.to_period("M")
)

# Count unique orders each month
monthly_orders = (
    order_trends.groupby("order_month")["Order Id"]
    .nunique()
    .sort_index()
)

print("\nMONTHLY ORDER TRENDS")
display(monthly_orders.to_frame("total_orders"))

# Plot monthly order trends
plt.figure(figsize=(12, 6))

plt.plot(
    monthly_orders.index.astype(str),
    monthly_orders.values,
    marker="o",
    linewidth=2,
    color="royalblue"
)

plt.title("Monthly E-Commerce Order Trends")
plt.xlabel("Month")
plt.ylabel("Number of Orders")

plt.xticks(rotation=90)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 15: Customer Order Analysis

# Use unique orders from Step 14
customer_orders = (
    order_trends.groupby("Customer Id")["Order Id"]
    .nunique()
)

# Customer statistics
total_customers = customer_orders.shape[0]
one_time_customers = (customer_orders == 1).sum()
repeat_customers = (customer_orders > 1).sum()

repeat_percentage = (
    repeat_customers / total_customers * 100
    if total_customers else 0
)

print("CUSTOMER ORDER ANALYSIS")
print("--------------------------")

print("Total Unique Customers:", total_customers)
print("One-Time Customers:", one_time_customers)
print("Repeat Customers:", repeat_customers)

print(
    f"Repeat Customer Percentage: {repeat_percentage:.2f}%"
)

print(
    "Average Orders Per Customer:",
    round(customer_orders.mean(), 2)
)

# Group customers by number of orders
customer_groups = pd.cut(
    customer_orders,
    bins=[0, 1, 3, 5, 10, float("inf")],
    labels=["1", "2-3", "4-5", "6-10", "11+"]
)

group_counts = (
    customer_groups.value_counts()
    .sort_index()
)

print("\nCUSTOMER ORDER DISTRIBUTION")
display(group_counts.to_frame("Customers"))

# Visualization
plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=group_counts.index.astype(str),
    y=group_counts.values,
    color="steelblue"
)

plt.title("Customer Order Frequency")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Customers")

for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(
        f"{int(height):,}",
        (
            bar.get_x() + bar.get_width() / 2,
            height
        ),
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()


In [ ]:

# STEP 16: Product Category Sales Analysis

# Identify category column
category_col = (
    "Category Name"
    if "Category Name" in df.columns
    else "Product Category Id"
)

# Check net sales column
if "Order Item Total" not in df.columns:
    raise KeyError(
        "Order Item Total column missing. "
        "Check available sales columns."
    )

# Use item-level records for sales analysis
sales_df = df.copy()

sales_df["Order Item Total"] = pd.to_numeric(
    sales_df["Order Item Total"],
    errors="coerce"
)

# Calculate category-wise sales
category_sales = (
    sales_df.groupby(category_col)
    .agg(
        total_sales=("Order Item Total", "sum"),
        order_items=("Order Item Id", "count")
    )
    .sort_values("total_sales", ascending=False)
    .head(10)
)

print("TOP 10 PRODUCT CATEGORIES BY SALES")
display(category_sales.round(2))

# Create visualization
plt.figure(figsize=(12, 6))

ax = sns.barplot(
    x=category_sales["total_sales"],
    y=category_sales.index.astype(str),
    color="steelblue"
)

plt.title("Top 10 Product Categories by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()


In [ ]:

# STEP 17: Product Category Profitability Analysis

profit_df = df.copy()

profit_df["Benefit per order"] = pd.to_numeric(
    profit_df["Benefit per order"],
    errors="coerce"
)

profit_df = profit_df.dropna(
    subset=["Benefit per order"]
)

# Analyze profitability indicators for top sales categories
category_profit = (
    profit_df.groupby(category_col)
    .agg(
        records=("Benefit per order", "size"),
        avg_benefit=("Benefit per order", "mean"),
        negative_benefit_pct=(
            "Benefit per order",
            lambda x: (x < 0).mean() * 100
        )
    )
    .reindex(category_sales.index)
)

print("CATEGORY PROFITABILITY ANALYSIS")
display(category_profit.round(2))

# Visualize negative benefit percentage
plt.figure(figsize=(12, 6))

ax = sns.barplot(
    x=category_profit["negative_benefit_pct"],
    y=category_profit.index.astype(str),
    color="indianred"
)

plt.title("Negative Benefit Rate by Product Category")
plt.xlabel("Records with Negative Benefit (%)")
plt.ylabel("Product Category")

for bar in ax.patches:
    width = bar.get_width()
    ax.annotate(
        f"{width:.1f}%",
        (width, bar.get_y() + bar.get_height() / 2),
        va="center",
        ha="left",
        xytext=(4, 0),
        textcoords="offset points"
    )

plt.xlim(
    0,
    max(10, category_profit["negative_benefit_pct"].max() * 1.2)
)

plt.tight_layout()
plt.show()


In [ ]:

# STEP 18: Regional Sales & Delivery Analysis

# Identify region column
region_col = next(
    (
        c for c in ["Order Region", "Order Country", "Order State"]
        if c in df.columns
    ),
    None
)

if region_col is None:
    raise KeyError("Region column not found.")

print("Using Region Column:", region_col)

# PART 1: Regional Sales Analysis
regional_sales_df = df.copy()

regional_sales_df["Order Item Total"] = pd.to_numeric(
    regional_sales_df["Order Item Total"],
    errors="coerce"
)

regional_sales = (
    regional_sales_df.groupby(region_col)["Order Item Total"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nTOP 10 REGIONS BY SALES")
display(regional_sales.to_frame("Total Sales").round(2))

# Sales Visualization
plt.figure(figsize=(12, 6))

sns.barplot(
    x=regional_sales.values,
    y=regional_sales.index,
    color="steelblue"
)

plt.title("Top 10 Regions by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Region")

plt.tight_layout()
plt.show()

# PART 2: Region-wise Delivery Performance
regional_delivery = (
    orders_df.groupby(region_col)
    .agg(
        total_orders=("Order Id", "nunique"),
        late_orders=("is_late", "sum")
    )
)

regional_delivery["late_rate"] = (
    regional_delivery["late_orders"] /
    regional_delivery["total_orders"] * 100
)

# Focus on the top 10 sales regions
regional_delivery = regional_delivery.reindex(
    regional_sales.index
)

print("\nREGIONAL DELIVERY PERFORMANCE")
display(regional_delivery.round(2))

# Late Delivery Visualization
plt.figure(figsize=(12, 6))

sns.barplot(
    x=regional_delivery["late_rate"],
    y=regional_delivery.index,
    color="indianred"
)

plt.title("Late Delivery Rate in Top Sales Regions")
plt.xlabel("Late Delivery Rate (%)")
plt.ylabel("Region")
plt.xlim(0, 100)

plt.tight_layout()
plt.show()


In [ ]:

# STEP 19: Correlation Analysis

# Select delivery-related numerical columns
selected_columns = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "processing_days",
    "delay_days",
    "is_late"
]

# Check available columns
available_columns = [
    col for col in selected_columns
    if col in orders_df.columns
]

# Use unique KPI-eligible orders
correlation_df = orders_df[
    available_columns
].copy()

# Convert columns to numeric
for col in available_columns:
    correlation_df[col] = pd.to_numeric(
        correlation_df[col],
        errors="coerce"
    )

# Calculate Spearman correlation
correlation_matrix = correlation_df.corr(
    method="spearman"
)

print("DELIVERY CORRELATION MATRIX")
display(correlation_matrix.round(2))

# Create correlation heatmap
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title("Delivery Performance Correlation Heatmap")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

# Correlations with late delivery
print("\nCORRELATION WITH LATE DELIVERY")

display(
    correlation_matrix["is_late"]
    .drop("is_late")
    .sort_values(ascending=False)
)


In [ ]:

# STEP 20: Data Redundancy Verification

# Select columns
check_df = orders_df[
    ["Days for shipping (real)", "processing_days"]
].copy()

# Convert to numeric
for col in check_df.columns:
    check_df[col] = pd.to_numeric(
        check_df[col],
        errors="coerce"
    )

# Exclude missing values
valid_df = check_df.dropna()

# Compare both columns
matching = (
    valid_df["Days for shipping (real)"]
    == valid_df["processing_days"]
).sum()

different = len(valid_df) - matching

print("DATA REDUNDANCY CHECK")
print("-------------------------")
print("Records Compared:", len(valid_df))
print("Matching Records:", matching)
print("Different Records:", different)

print(
    "Match Percentage:",
    round(matching / len(valid_df) * 100, 2)
    if len(valid_df) else 0,
    "%"
)

print("\nSAMPLE COMPARISON")
display(valid_df.head(10))


In [ ]:

# STEP 21: Generate Final EDA Findings Report

from pathlib import Path

# Locate existing EDA folder
project_root = file_path.resolve().parents[2]
eda_folder = project_root / "EDA (Python-scripts)"

eda_folder.mkdir(parents=True, exist_ok=True)

# Calculate summary metrics
total_orders = len(orders_df)
late_orders = int(orders_df["is_late"].sum())
late_rate = late_orders / total_orders * 100

avg_delay = late_df["delay_days"].mean()

total_customers = len(customer_orders)
repeat_customers = int((customer_orders > 1).sum())
repeat_rate = repeat_customers / total_customers * 100

highest_risk_region = regional_delivery["late_rate"].idxmax()
highest_region_rate = regional_delivery["late_rate"].max()

top_category = category_sales.index[0]

# Prepare report
report = f"""
# E-Commerce & Logistics Performance Analysis
## Exploratory Data Analysis Findings

### 1. Dataset Overview
- Total Records: {len(df):,}
- Total Columns: {len(df.columns)}
- KPI-Eligible Unique Orders: {total_orders:,}

### 2. Delivery Performance
- Late Orders: {late_orders:,}
- Late Delivery Rate: {late_rate:.2f}%
- Average Delay Among Late Orders: {avg_delay:.2f} days

### 3. Shipping Mode Analysis
- First Class Late Rate: 100%
- Second Class Late Rate: 80.09%
- Same Day Late Rate: 48.51%
- Standard Class Late Rate: 39.76%

### 4. Customer Analysis
- Total Unique Customers: {total_customers:,}
- Repeat Customers: {repeat_customers:,}
- Repeat Customer Rate: {repeat_rate:.2f}%

### 5. Product Category Analysis
- Highest Sales Category: {top_category}

### 6. Regional Delivery Analysis
- Highest Late Rate Among Top Sales Regions:
  {highest_risk_region} ({highest_region_rate:.2f}%)

### 7. Data Redundancy Findings
- processing_days matches actual shipping days
  in all 28,965 compared eligible unique orders.
- Avoid treating both fields as independent features.

### 8. Business Recommendations
1. Investigate First Class and Second Class delivery targets.
2. Improve on-time delivery performance.
3. Investigate regions with higher late delivery rates.
4. Review categories with negative benefit records.
5. Analyze repeat purchasing and customer retention.

### 9. Analysis Notes
- Delivery KPIs use unique KPI-eligible orders.
- Sales analysis uses order-item records.
- Correlation does not establish causation.
- Shipping-mode results should be checked against
  scheduled delivery definitions.
"""

# Save Markdown report
report_path = eda_folder / "EDA_Findings.md"

report_path.write_text(report, encoding="utf-8")

print("EDA Findings Report Generated Successfully!")
print("Saved Location:", report_path)


In [ ]:

# STEP 22: Create Visualization Folder

from pathlib import Path

visualizations_dir = (
    project_root / "EDA (Python-scripts)" / "visualizations"
)

visualizations_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Visualization folder created successfully!")
print("Location:", visualizations_dir)


In [ ]:

# STEP 24: AUTOMATIC EXPORT OF ALL REMAINING EDA GRAPHS

import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Create or locate visualization folder
visualizations_dir = (
    Path(file_path).resolve().parents[2]
    / "EDA (Python-scripts)"
    / "visualizations"
)

visualizations_dir.mkdir(parents=True, exist_ok=True)

# Helper function for exporting charts
def save_chart(filename):
    plt.tight_layout()
    plt.savefig(
        visualizations_dir / filename,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.close()
    print(f"Saved: {filename}")


# -------------------------------------------------
# GRAPH 02: SHIPPING MODE ANALYSIS
# -------------------------------------------------

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=shipping_analysis,
    x="Shipping Mode",
    y="late_percentage",
    color="steelblue"
)

plt.title("Late Delivery Rate by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Late Delivery Rate (%)")
plt.ylim(0, 110)

for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(
        f"{height:.2f}%",
        (bar.get_x() + bar.get_width() / 2, height),
        ha="center",
        va="bottom"
    )

save_chart("02_shipping_mode_analysis.png")


# -------------------------------------------------
# GRAPH 03: DELIVERY DELAY DISTRIBUTION
# -------------------------------------------------

plt.figure(figsize=(10, 6))

sns.histplot(
    data=late_df,
    x="delay_days",
    discrete=True,
    color="steelblue"
)

plt.title("Distribution of Delivery Delays")
plt.xlabel("Delivery Delay (Days)")
plt.ylabel("Number of Late Orders")

save_chart("03_delivery_delay_distribution.png")


# -------------------------------------------------
# GRAPH 04: ACTUAL VS SCHEDULED SHIPPING
# -------------------------------------------------

shipping_days[
    ["avg_actual_days", "avg_scheduled_days"]
].plot(
    kind="bar",
    figsize=(10, 6),
    color=["steelblue", "orange"]
)

plt.title("Actual vs Scheduled Shipping Days")
plt.xlabel("Shipping Mode")
plt.ylabel("Average Shipping Days")
plt.xticks(rotation=0)
plt.legend(["Actual Days", "Scheduled Days"])

save_chart("04_actual_vs_scheduled.png")


# -------------------------------------------------
# GRAPH 05: MONTHLY ORDER TRENDS
# -------------------------------------------------

plt.figure(figsize=(13, 6))

plt.plot(
    monthly_orders.index.astype(str),
    monthly_orders.values,
    marker="o",
    color="royalblue",
    linewidth=2
)

plt.title("Monthly E-Commerce Order Trends")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=90)
plt.grid(alpha=0.3)

save_chart("05_monthly_order_trends.png")


# -------------------------------------------------
# GRAPH 06: CUSTOMER ORDER ANALYSIS
# -------------------------------------------------

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    x=group_counts.index.astype(str),
    y=group_counts.values,
    color="steelblue"
)

plt.title("Customer Order Frequency")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Customers")

for bar in ax.patches:
    height = bar.get_height()
    ax.annotate(
        f"{int(height):,}",
        (bar.get_x() + bar.get_width() / 2, height),
        ha="center",
        va="bottom"
    )

save_chart("06_customer_analysis.png")


# -------------------------------------------------
# GRAPH 07: PRODUCT CATEGORY SALES
# -------------------------------------------------

plt.figure(figsize=(12, 7))

sns.barplot(
    x=category_sales["total_sales"],
    y=category_sales.index.astype(str),
    color="steelblue"
)

plt.title("Top 10 Product Categories by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product Category")

save_chart("07_category_sales.png")


# -------------------------------------------------
# GRAPH 08: PROFITABILITY ANALYSIS
# -------------------------------------------------

plt.figure(figsize=(12, 7))

sns.barplot(
    x=category_profit["negative_benefit_pct"],
    y=category_profit.index.astype(str),
    color="indianred"
)

plt.title("Negative Benefit Rate by Product Category")
plt.xlabel("Records with Negative Benefit (%)")
plt.ylabel("Product Category")

save_chart("08_profitability_analysis.png")


# -------------------------------------------------
# GRAPH 09: REGIONAL SALES
# -------------------------------------------------

plt.figure(figsize=(12, 7))

sns.barplot(
    x=regional_sales.values,
    y=regional_sales.index.astype(str),
    color="steelblue"
)

plt.title("Top 10 Regions by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Region")

save_chart("09_regional_sales.png")


# -------------------------------------------------
# GRAPH 10: REGIONAL DELIVERY PERFORMANCE
# -------------------------------------------------

plt.figure(figsize=(12, 7))

sns.barplot(
    x=regional_delivery["late_rate"],
    y=regional_delivery.index.astype(str),
    color="indianred"
)

plt.title("Late Delivery Rate in Top Sales Regions")
plt.xlabel("Late Delivery Rate (%)")
plt.ylabel("Region")
plt.xlim(0, 100)

save_chart("10_regional_delivery.png")


# -------------------------------------------------
# GRAPH 11: CORRELATION HEATMAP
# -------------------------------------------------

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title("Delivery Performance Correlation Heatmap")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

save_chart("11_correlation_heatmap.png")


# -------------------------------------------------
# FINAL VERIFICATION
# -------------------------------------------------

print("\n" + "=" * 45)
print("EDA VISUALIZATION EXPORT COMPLETED")
print("=" * 45)

all_graphs = sorted(visualizations_dir.glob("*.png"))

print(f"\nTotal PNG Graphs: {len(all_graphs)}")

for graph in all_graphs:
    print("✓", graph.name)

print("\nSaved Location:")
print(visualizations_dir)
